**Fine-Tuning Whisper-Tiny for Assamese Speech Recognition using Common Voice**

In this project, we fine-tune OpenAI’s Whisper-Tiny model for automatic speech recognition (ASR) on the Assamese language using the Mozilla Common Voice 11.0 dataset.

Whisper is a powerful sequence-to-sequence speech recognition model developed by OpenAI, capable of handling multilingual and multitask transcription. We're leveraging Hugging Face’s transformers and datasets libraries to:

Preprocess audio using log-Mel spectrograms via WhisperFeatureExtractor.

Tokenize transcripts in Assamese using WhisperTokenizer.

Utilize WhisperProcessor to combine preprocessing and tokenization seamlessly.

Train with Seq2SeqTrainer, optimizing for Word Error Rate (WER).

Apply efficient training with fp16, gradient checkpointing, and a small batch size, making it feasible to run even on modest GPU setups.

✅ Goals:
Adapt Whisper-Tiny to transcribe spoken Assamese.

Evaluate performance using WER (Word Error Rate).

Showcase how compact models like Whisper-Tiny can be fine-tuned for low-resource languages.

This notebook is a practical example of how to bring multilingual ASR to underrepresented languages using open-source tools and pretrained models.

In [ ]:
!pip install datasets transformers evaluate jiwer

In [ ]:
import torch
import datasets
import evaluate
import numpy as np
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from transformers import  TrainingArguments, Trainer


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)  # Move model to GPU


In [ ]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("mozilla-foundation/common_voice_11_0")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/14.4k [00:00<?, ?B/s]

common_voice_11_0.py:   0%|          | 0.00/8.13k [00:00<?, ?B/s]

languages.py:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

release_stats.py:   0%|          | 0.00/60.9k [00:00<?, ?B/s]

The repository for mozilla-foundation/common_voice_11_0 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/mozilla-foundation/common_voice_11_0.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


In [ ]:
print(configs)

['en', 'fa', 'fr', 'es', 'sl', 'kab', 'cy', 'ca', 'de', 'tt', 'ta', 'ru', 'nl', 'it', 'eu', 'tr', 'ar', 'zh-TW', 'br', 'pt', 'eo', 'zh-CN', 'id', 'ia', 'lv', 'ja', 'rw', 'sv-SE', 'cnh', 'et', 'ky', 'ro', 'hsb', 'el', 'cs', 'pl', 'rm-sursilv', 'rm-vallader', 'mn', 'zh-HK', 'ab', 'cv', 'uk', 'mt', 'as', 'ka', 'fy-NL', 'dv', 'pa-IN', 'vi', 'or', 'ga-IE', 'fi', 'hu', 'th', 'lt', 'lg', 'hi', 'bas', 'sk', 'kmr', 'bg', 'kk', 'ba', 'gl', 'ug', 'hy-AM', 'be', 'ur', 'gn', 'sr', 'uz', 'mr', 'da', 'myv', 'nn-NO', 'ha', 'ckb', 'ml', 'mdf', 'sw', 'sat', 'tig', 'ig', 'nan-tw', 'mhr', 'bn', 'tok', 'yue', 'sah', 'mk', 'sc', 'skr', 'ti', 'mrj', 'tw', 'vot', 'az', 'ast', 'ne-NP']


In [ ]:
train_dataset = datasets.load_dataset("mozilla-foundation/common_voice_11_0","as",split="train")

In [ ]:
test_dataset=datasets.load_dataset("mozilla-foundation/common_voice_11_0","as",split="test[:10]")

In [ ]:
print(train_dataset[0])

{'client_id': 'af73187438537bf78a33930717694a696d489072f8e334e9a21dd46fa09ae9c3040e4d44e97e8c2bea2bfda5d74e73063f486d36ca84f4bfc56b43a58bb9389b', 'path': '/root/.cache/huggingface/datasets/downloads/extracted/fdcfd174c1db561f74a5aab292ff32458ceffd67c10de1ac5f5b77eae211090c/as_train_0/common_voice_as_22074894.mp3', 'audio': {'path': '/root/.cache/huggingface/datasets/downloads/extracted/fdcfd174c1db561f74a5aab292ff32458ceffd67c10de1ac5f5b77eae211090c/as_train_0/common_voice_as_22074894.mp3', 'array': array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
       -1.51620850e-06, -1.28747206e-06, -6.44893703e-07]), 'sampling_rate': 48000}, 'sentence': 'দেখিলে যে অসমীয়া মানুহৰ জ্ঞান-উন্নতি পিনে অলপাে মনকাণ নাই', 'up_votes': 2, 'down_votes': 0, 'age': '', 'gender': '', 'accent': '', 'locale': 'as', 'segment': ''}


In [ ]:
from transformers import WhisperFeatureExtractor

In [ ]:
feature_whisp=WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")

In [ ]:
from transformers import WhisperTokenizer

In [ ]:
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", language="as", task="transcribe")

In [ ]:
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="as", task="transcribe")

In [ ]:
from datasets import Audio

In [ ]:
audio_input=train_dataset.cast_column("audio",Audio(sampling_rate=16000))

In [ ]:
print(audio_input["audio"][0])

{'path': '/root/.cache/huggingface/datasets/downloads/extracted/fdcfd174c1db561f74a5aab292ff32458ceffd67c10de1ac5f5b77eae211090c/as_train_0/common_voice_as_22074894.mp3', 'array': array([ 5.29395592e-23, -6.61744490e-23,  1.48892510e-22, ...,
        1.20360312e-07, -1.29233990e-06, -1.51768404e-06]), 'sampling_rate': 16000}


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_whisp(audio["array"], sampling_rate=16000).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [ ]:
latest_train_dataset = train_dataset.map(prepare_dataset,remove_columns=["audio"])
latest_test_dataset = test_dataset.map(prepare_dataset,remove_columns=["audio"])

In [ ]:
print(test_dataset[0])

{'client_id': '256a0ab2364a69f69c073cac0258702b33d8269739cd421abc11e45a2c7678787e46498a06ee3a13077e9dfea487f22061fc549def882ff192ca5fb1d2671807', 'path': '/root/.cache/huggingface/datasets/downloads/extracted/03ae5299b73997977a2e97df770578f66fc446cf00fee784e3ba200e714b189b/as_test_0/common_voice_as_33313041.mp3', 'audio': {'path': '/root/.cache/huggingface/datasets/downloads/extracted/03ae5299b73997977a2e97df770578f66fc446cf00fee784e3ba200e714b189b/as_test_0/common_voice_as_33313041.mp3', 'array': array([9.94759830e-14, 1.80477855e-12, 3.75166564e-12, ...,
       1.38468531e-04, 3.25373490e-04, 2.74217513e-04]), 'sampling_rate': 48000}, 'sentence': 'দঢ়াই দঢ়াই মাতি গৈছিল সকলোকে।', 'up_votes': 2, 'down_votes': 0, 'age': '', 'gender': '', 'accent': '', 'locale': 'as', 'segment': ''}


In [ ]:
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)

In [ ]:
model.config.forced_decoder_ids = None
model.generation_config.language = "assamese"
model.generation_config.task = "transcribe"

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
    """

    processor: WhisperProcessor
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need
        # different padding methods
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels


        return batch

In [ ]:
data_collator = DataCollatorCTCWithPadding(processor=processor, decoder_start_token_id=model.config.decoder_start_token_id,)

In [ ]:
wer_metric = evaluate.load("wer")

In [ ]:
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    # we do not want to group tokens when computing the metrics
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [ ]:
from transformers import Seq2SeqTrainingArguments

In [ ]:

training_args = Seq2SeqTrainingArguments(
    output_dir="./out",  # change to a repo name of your choice
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=100,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to=None,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=latest_train_dataset,
    eval_dataset=latest_test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


<ipython-input-33-1226408ea6c6>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 22f1001470 (22f1001470-iit-madras-alumni-association) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Step,Training Loss,Validation Loss,Wer
100,1.434000,1.449646,1.000000


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 

TrainOutput(global_step=100, training_loss=1.847235107421875, metrics={'train_runtime': 224.0682, 'train_samples_per_second': 3.57, 'train_steps_per_second': 0.446, 'total_flos': 1.9695108096e+16, 'train_loss': 1.847235107421875, 'epoch': 0.970873786407767})